# Notebook 01 — Seller Simulation

Study A. Synthesize virtual sellers from the real test-set product-name pool.
The design follows the first paper's store-simulation logic (dominant-category
ratio manipulation with ground-truth defined as the dominant category) and
extends it to the seller level (multiple stores per seller) while adding a
category-difficulty tilt so that seller reliability is genuinely heterogeneous.

Manipulated factors:
- dominant-category ratio regime: focused / cross-category / diversified
- number of stores per seller (1 recovers the first-paper single-store case)
- number of items N per seller
- difficulty tilt: easy / neutral / hard specialization
- disagreement method: A (natural) and B (noise-injected), for RQ2 stress

Each seller carries item-level predictions/confidence/correctness needed for
seller-level SCS in the next notebook, plus a ground-truth segment.


In [1]:
# %% ============================================================
# Notebook 01 — Seller Simulation
# Imports, paths, reproducibility
# ============================================================
import os
import json
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
ARTIFACT_DIR = os.path.join(ROOT, "artifacts")
TAB_DIR = os.path.join(ROOT, "results", "tables")

items = pd.read_csv(os.path.join(ARTIFACT_DIR, "test_predictions.csv"))
proba = np.load(os.path.join(ARTIFACT_DIR, "test_proba.npy"))
N_CLASSES = proba.shape[1]
print("Loaded", items.shape[0], "items,", N_CLASSES, "classes")
print(items.head())

# %% ============================================================
# Index the item pool by true label so a store can realize a target
# dominant-category ratio by drawing dominant vs. non-dominant items.
# ============================================================
items = items.reset_index(drop=True)
items["item_id"] = items.index
pool_by_label = {c: items.index[items["label"] == c].to_numpy()
                 for c in range(N_CLASSES)}
for c in range(N_CLASSES):
    print(f"label {c:2d}: {len(pool_by_label[c])} items in pool")

# %% ============================================================
# Category classification difficulty. Real categories differ in how
# hard they are to classify; ranking them lets a seller specialize in
# easy or hard categories, giving seller reliability real spread.
# ============================================================
cat_acc = items.groupby("label")["correct"].mean().sort_values()
easy_cats = cat_acc.index[cat_acc.values >= cat_acc.median()].to_numpy()
hard_cats = cat_acc.index[cat_acc.values < cat_acc.median()].to_numpy()
print("Category accuracy (ascending):")
print(cat_acc.round(3).to_dict())
print("hard categories:", hard_cats.tolist())
print("easy categories:", easy_cats.tolist())

# %% ============================================================
# Simulation design grid.
# Dominant-ratio regimes follow the first paper's manipulation:
# a store's dominant category holds a share drawn from the regime's
# uniform interval, and the remaining share is spread over other
# categories. The ground-truth Seg is the dominant category.
# ============================================================
DOMINANT_REGIMES = {
    "focused":      (0.6, 1.0),   # first-paper baseline regime
    "cross":        (0.3, 0.6),   # cross-category mixing
    "diversified":  (0.1, 0.3),   # extreme mixing, weak signal
}
STORE_COUNTS = [1, 2, 3, 5]
ITEM_COUNTS = [30, 100, 300]
N_SELLERS_PER_CELL = 40

design_cells = [(r, s, n)
                for r in DOMINANT_REGIMES
                for s in STORE_COUNTS
                for n in ITEM_COUNTS]
print("Design cells:", len(design_cells))
print("Sellers per method:", len(design_cells) * N_SELLERS_PER_CELL)

# %% ============================================================
# Store synthesis under a target dominant-category ratio.
# A dominant category is chosen (tilted toward easy/hard categories);
# a share p_dom (drawn from the regime interval) of items is drawn from
# the dominant category, and the remaining items are spread over other
# categories. Drawing from the real pool preserves each item's true
# prediction / confidence / correctness.
# ============================================================
def pick_dominant(tilt, rng):
    if tilt == "easy":
        return int(rng.choice(easy_cats))
    if tilt == "hard":
        return int(rng.choice(hard_cats))
    return int(rng.integers(0, N_CLASSES))


def synthesize_store(regime, n_items, rng, tilt="neutral", dom_cat=None):
    lo, hi = DOMINANT_REGIMES[regime]
    p_dom = rng.uniform(lo, hi)
    if dom_cat is None:
        dom_cat = pick_dominant(tilt, rng)
    n_dom = int(round(n_items * p_dom))
    n_rest = n_items - n_dom

    chosen = []
    if n_dom > 0:
        chosen.append(rng.choice(pool_by_label[dom_cat], size=n_dom, replace=True))
    if n_rest > 0:
        others = [c for c in range(N_CLASSES) if c != dom_cat]
        n_other_cats = int(rng.integers(1, min(5, len(others)) + 1))
        other_cats = rng.choice(others, size=n_other_cats, replace=False)
        split = rng.multinomial(n_rest, np.ones(n_other_cats) / n_other_cats)
        for c, k in zip(other_cats, split):
            if k > 0:
                chosen.append(rng.choice(pool_by_label[int(c)], size=int(k), replace=True))
    idx = np.concatenate(chosen) if chosen else np.array([], dtype=int)
    return idx, dom_cat


_idx, _dc = synthesize_store("focused", 30, rng, tilt="hard")
print("sample store size:", len(_idx), "dominant cat:", _dc,
      "->", items.loc[_idx, "label"].value_counts().to_dict())

# %% ============================================================
# Seller synthesis. A seller spans n_stores stores that SHARE one
# ground-truth dominant category (the seller's business domain), but
# each store draws its own dominant ratio, so cross-store disagreement
# can still arise. Method B injects label noise to stress the
# between-store term for RQ2.
# ============================================================
def synthesize_seller(regime, n_stores, n_items, rng, eta=0.0, tilt="neutral"):
    dom_cat = pick_dominant(tilt, rng)   # seller-level ground truth
    per_store = max(5, n_items // n_stores)
    records = []
    for store_id in range(n_stores):
        idx, _ = synthesize_store(regime, per_store, rng, tilt=tilt, dom_cat=dom_cat)
        if len(idx) == 0:
            continue
        sub = items.loc[idx, ["item_id", "label", "pred", "confidence", "correct"]].copy()
        sub = sub.reset_index(drop=True)
        sub["store_id"] = store_id
        if eta > 0:
            pred_arr = sub["pred"].to_numpy().copy()
            label_arr = sub["label"].to_numpy()
            flip = rng.random(len(sub)) < eta
            n_flip = int(flip.sum())
            if n_flip > 0:
                pred_arr[flip] = rng.integers(0, N_CLASSES, size=n_flip)
            sub["pred"] = pred_arr
            sub["correct"] = (pred_arr == label_arr).astype(int)
        records.append(sub)
    if not records:
        return None, dom_cat
    return pd.concat(records, ignore_index=True), dom_cat


_s, _dc = synthesize_seller("cross", 3, 100, rng, tilt="neutral")
print("sample seller items:", len(_s), "stores:", _s["store_id"].nunique(),
      "ground-truth cat:", _dc)

# %% ============================================================
# Run the full grid for both disagreement methods.
# Method A: eta = 0 (natural classifier disagreement).
# Method B: eta = 0.15 (noise-injected) for the between-store stress.
# The seller's ground-truth Seg is the dominant category; seller
# accuracy is measured against the true labels of drawn items.
# ============================================================
TILTS = ["easy", "neutral", "hard"]


def run_grid(eta, tag, rng):
    rows = []
    seller_items = []
    uid = 0
    for (regime, n_stores, n_items) in design_cells:
        for k in range(N_SELLERS_PER_CELL):
            tilt = TILTS[k % len(TILTS)]
            seller, dom_cat = synthesize_seller(regime, n_stores, n_items, rng,
                                                eta=eta, tilt=tilt)
            if seller is None or seller.empty:
                continue
            seller["seller_id"] = uid
            rows.append({
                "seller_id": uid,
                "regime": regime,
                "n_stores": n_stores,
                "n_items_target": n_items,
                "n_items_actual": len(seller),
                "method": tag,
                "eta": eta,
                "tilt": tilt,
                "gt_segment": dom_cat,
                "seller_accuracy": seller["correct"].mean(),
            })
            seller_items.append(seller)
            uid += 1
    return pd.DataFrame(rows), pd.concat(seller_items, ignore_index=True)


meta_A, items_A = run_grid(0.0, "A_real", rng)
meta_B, items_B = run_grid(0.15, "B_noise", rng)
print("Method A sellers:", meta_A.shape[0], "| items:", items_A.shape[0])
print("Method B sellers:", meta_B.shape[0], "| items:", items_B.shape[0])

# %% ============================================================
# Persist simulation outputs (offset method-B ids before concat).
# ============================================================
offset = meta_A["seller_id"].max() + 1
meta_B = meta_B.copy(); items_B = items_B.copy()
meta_B["seller_id"] += offset
items_B["seller_id"] += offset

seller_meta = pd.concat([meta_A, meta_B], ignore_index=True)
seller_items = pd.concat([items_A, items_B], ignore_index=True)
seller_meta.to_csv(os.path.join(ARTIFACT_DIR, "seller_meta.csv"), index=False)
seller_items.to_csv(os.path.join(ARTIFACT_DIR, "seller_items.csv"), index=False)

print("Total sellers:", seller_meta.shape[0])
print("Total seller-item rows:", seller_items.shape[0])
print()
print("Method A seller accuracy by dominant-ratio regime:")
print(meta_A.groupby("regime")["seller_accuracy"].agg(["mean", "std", "min", "max"]).round(3))

# %% ============================================================
# Simulation design realization table.
# ============================================================
summary = seller_meta.groupby(["method", "regime", "n_stores"]).agg(
    n_sellers=("seller_id", "size"),
    mean_items=("n_items_actual", "mean"),
    mean_acc=("seller_accuracy", "mean"),
).reset_index()
summary.to_csv(os.path.join(TAB_DIR, "table_simulation_design.csv"), index=False)
print(summary.to_string(index=False))

Loaded 22011 items, 11 classes
                                  prod_name  label  pred  confidence  correct
0                슬림콘솔 폭좁은 대리석 장식장 미니 테이블 복도      0     0    0.991818        1
1  세라믹 거실테이블 원형 서랍 화이트 티테이블 고급스러운 소파 테이블 세트      0     0    0.993393        1
2                    시베리아 차가버섯 추출분말 35g  6병      5     5    0.996376        1
3                       리얼룩 갤럭시S7엣지 롤리팝 케이스      2     2    0.994609        1
4          2022 아이폰SE3 3세대 투명 듀얼라인 카드 2장 젤리      2     2    0.994663        1
label  0: 2001 items in pool
label  1: 2001 items in pool
label  2: 2001 items in pool
label  3: 2001 items in pool
label  4: 2001 items in pool
label  5: 2001 items in pool
label  6: 2001 items in pool
label  7: 2001 items in pool
label  8: 2001 items in pool
label  9: 2001 items in pool
label 10: 2001 items in pool
Category accuracy (ascending):
{3: 0.704, 4: 0.81, 7: 0.814, 0: 0.826, 9: 0.883, 2: 0.89, 6: 0.892, 8: 0.926, 10: 0.942, 1: 0.958, 5: 0.96}
hard categories: [3, 4, 7, 0, 9]
easy cate